In [0]:
%sql

create or replace table workspace.silver.subscriptions
comment '''
    one row per subscription. notes: 
    (1) a home can have several subscriptions with a gap between them (home_002: no contract 2026-09-10 to 2026-10-01, but readings continue). consumption in a gap is shown without cost, handled in gold. 
    (2) fixed_price is in the source unit (ore/kwh, nok/mwh): normalised to main currency per kwh, same rule as spot prices. 
    (3) subscriptions can end mid-month: the full monthly fee is assumed, not prorated. open: whether ended_at is inclusive.
'''
as

select
    s.subscription_id,
    s.home_id,
    cast(s.started_at as date) as started_at,
    cast(s.ended_at as date) as ended_at,
    s.tariff_type,
    cast(s.monthly_fee as decimal(10, 2)) as monthly_fee,
    cast(cast(s.fixed_price as double) / case lower(s.currency)
        when 'nok' then 1000
        when 'sek' then 100
        else raise_error(concat_ws(' ', 'unexpected currency:', s.currency))
    end as decimal(12, 5)) as fixed_price_per_kwh,
    s.currency,
    current_timestamp() as _processed_at
from 
    workspace.bronze.subscriptions as s

In [0]:
%sql
select
    cast(m.measured_at_local as date) as measured_date,
    count(*) as row_count,
    sum(m.kwh) as kwh
from workspace.silver.meter_readings as m
where m.home_id = 'home_002'
    and m.measured_at_local >= '2026-09-10'
    and m.measured_at_local < '2026-10-01'
group by all
order by measured_date